# Build test_v2 (CPU): a fresh test set, not opened here

Draws 3,000 drawings per class from the Kaggle competition files that split v1 never used
(`configs/test_v2.yaml`, rules in `src/quickdraw_exp/test_v2.py`) and writes them in the layout of split v1.
**No model reads them here.** They are opened once, later, by the final evaluation final_v2.

**Run it as a separate notebook** (for example `ai-pictionary-build-test-v2`), not as a version of AI_pictionary.

**Before running:**
1. **Input**: the competition **Quick, Draw! Doodle Recognition Challenge** (Add Input → Competitions; accept its
   rules if Kaggle asks) and your Kaggle Dataset with the processed data (`dataset-ai-pictionary`).
2. **Accelerator: None** (CPU), **Internet on** (for the code).
3. **Save Version → Save & Run All**. Expected: 20-60 minutes.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping")

folders = [p.parent for p in Path("/kaggle/input").rglob("train_simplified") if p.is_dir()]
if not folders:
    raise RuntimeError("train_simplified not found: add the competition Quick, Draw! Doodle Recognition Challenge as input")
KAGGLE = folders[0]
n = len(list((KAGGLE / "train_simplified").glob("*.csv")))
print("competition files:", KAGGLE, n, "classes")


In [ ]:
OUT = Path("/kaggle/working/test_v2")
!PYTHONPATH=src python -m quickdraw_exp.test_v2 --kaggle "{KAGGLE}" --out {OUT}
print(sorted(p.name for p in OUT.iterdir()))
